# EasyMLOps LLM 决策模型 Demo（类 Jev 生态）

类 Jev 决策模型（DecisionModelPipe）+ LoRA 微调（LoRAFineTuner）端到端示例。
本 notebook 使用 **mock 决策引擎与 tiny 因果 LM**（CPU 离线可跑）；
真实使用时把引擎换成 `laya`（421M 本地）或 Jev 兼容服务即可，接口完全一致。

In [1]:
import sys, os
try:
    import easymlops
except ImportError:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
    import easymlops

import numpy as np
import pandas as pd
from easymlops.llm import DecisionModelPipe, DecisionSampleGenerator, LoRAFineTuner
print("easymlops:", easymlops.__file__)

easymlops: d:/datas/share/traes/easymlops-trae\easymlops\__init__.py


## 1. 决策数据与题目定义

Jev 契约：state（文本）+ typed questions（choice/score/noul）→ 校准概率。

In [2]:
df = pd.DataFrame({
    "text": ["我要退款", "物流太慢了", "你们服务真差", "查询订单", "东西坏了要退",
             "什么时候到货", "客服态度恶劣", "帮我查下快递"],
    "label": ["refund", "logistics", "other", "logistics", "refund",
              "logistics", "other", "logistics"],
})
QUESTIONS = [
    {"name": "intent", "type": "choice", "instructions": "用户意图",
     "criteria": {"refund": "退款退货", "logistics": "物流查询", "other": "其他"}},
    {"name": "is_complaint", "type": "noul", "instructions": "是否投诉/不满语气"},
]
df.head()

## 2. DecisionModelPipe 作为特征工程器

决策概率列可直接作为 LGBM 等表格模型的特征（本例用 mock 引擎演示契约）。

In [3]:
from easymlops.llm import DecisionModelPipe

class MockDecisionEngine(object):
    """mock: 真实场景换 laya Router 或 Jev 兼容服务, 接口一致"""
    def __init__(self, keywords):
        self.keywords = keywords
    def predict(self, state, questions, **kwargs):
        probs = {opt: 0.0 for opt in questions["intent"]["criteria"]}
        hit = "other"
        for kw, opt in self.keywords.items():
            if kw in state:
                hit = opt
                break
        probs[hit] = 0.9
        p_angry = 0.9 if any(w in state for w in ["差", "恶劣"]) else 0.1
        return {"answers": {"intent": {"choice": hit}, "is_complaint": {"p": p_angry}},
                "probabilities": {"intent": probs, "is_complaint": p_angry}}

engine = MockDecisionEngine({"退款": "refund", "退": "refund", "物流": "logistics",
                             "快递": "logistics", "到货": "logistics"})
pipe = DecisionModelPipe(questions=QUESTIONS, backend="laya", state_col="text")
pipe.fit(df)
pipe._engine = engine
feats = pipe.transform(df.copy())
print("决策特征列:", [c for c in feats.columns if c != "state"])
print(feats[["intent_pred", "intent_refund", "intent_logistics", "is_complaint_p"]].head(3))

决策特征列: ['intent_pred', 'intent_refund', 'intent_logistics', 'intent_other', 'is_complaint_p']
  intent_pred  intent_refund  intent_logistics  is_complaint_p
0      refund            0.9               0.0             0.1
1   logistics            0.0               0.9             0.1
2       other            0.0               0.0             0.9


## 3. 决策特征 → 表格模型

概率列与表格特征拼接送入 LGBM（数值列直接可用）。

In [4]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score

model_df = df.copy()
model_df["intent_refund"] = feats["intent_refund"].values
model_df["intent_logistics"] = feats["intent_logistics"].values
model_df["is_complaint_p"] = feats["is_complaint_p"].values
model_df["text_len"] = model_df["text"].str.len()

train_X = model_df[["intent_refund", "intent_logistics", "is_complaint_p", "text_len"]]
model = lgb.LGBMClassifier(n_estimators=20, verbosity=-1, random_state=42)
model.fit(train_X, df["label"])
pred = model.predict(train_X)
print("训练集准确率:", round(accuracy_score(df["label"], pred), 3))

训练集准确率: 0.5


## 4. 微调数据流水线：表格 → Jev 契约样本

DecisionSampleGenerator 从带标签表格生成微调样本。

In [5]:
from easymlops.llm import DecisionSampleGenerator

gen = DecisionSampleGenerator(
    questions=[{"name": "intent", "type": "choice", "instructions": "用户意图"}],
    text_col="text", random_state=42)
samples = gen.generate(df, label_map={"intent": "label"})
train_samples, val_samples = gen.split(samples, val_ratio=0.25)
print("train/val:", len(train_samples), "/", len(val_samples))
print("样本示例:", {k: str(v)[:60] for k, v in train_samples[0].items()})

train/val: 6 / 2
样本示例: {'state': '客服态度恶劣', 'questions': "{'intent': {'type': 'choice', 'instructions': '用户意图', 'crite", 'answers': "{'intent': {'choice': 'other'}}"}


## 5. LoRA 微调（tiny 因果 LM, CPU 真实训练）

决策样本转为指令文本后 LoRA SFT；真实场景把 model 换成 OneJev/Jeff/Laya 骨干。

In [6]:
import json
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from transformers import GPT2Config, GPT2LMHeadModel

# 构建一个 tiny 因果 LM(仅演示, ~2MB)
TINY = "_tiny_lm_demo"
if not os.path.isdir(TINY):
    cfg = GPT2Config(vocab_size=256, n_positions=64, n_embd=32, n_layer=1, n_head=1)
    GPT2LMHeadModel(cfg).save_pretrained(TINY)
    # 最小 tokenizer 文件(legacy GPT2 格式)
    vocab = {"<|endoftext|>": 0}
    for i in range(33, 127):
        vocab[chr(i)] = i - 32
    with open(os.path.join(TINY, "vocab.json"), "w", encoding="utf-8") as f:
        json.dump(vocab, f)
    with open(os.path.join(TINY, "merges.txt"), "w", encoding="utf-8") as f:
        f.write("#version: 0.2\n")

tuner = LoRAFineTuner("tiny-lm", local_path=TINY, lora_rank=4, lora_alpha=8, device="cpu")
info = tuner.auto_adapt()
print("自动适配:", info["model_type"], "| targets:", info["target_modules"])

sft = [{"prompt": "state: {}\nquestion: 用户意图\n".format(s["state"]),
        "completion": "answer: {}\n".format(s["answers"]["intent"]["choice"])}
       for s in samples]
result = tuner.train_sft(sft, output_dir="_demo_adapter", epochs=2, lr=5e-4, batch_size=2)
print("LoRA 训练完成: steps={} final_loss={}".format(result["steps"], result["final_loss"]))

自动适配: gpt2 | targets: ['c_proj']
LoRA 训练完成: steps=8 final_loss=5.599294662475586


In [7]:
# adapter 合并导出(可再用 vLLM/ollama 服务化)
merged = tuner.merge_and_export("_demo_merged")
print("merged model at:", merged)
import shutil
shutil.rmtree("_demo_adapter", ignore_errors=True)
shutil.rmtree("_demo_merged", ignore_errors=True)
shutil.rmtree(TINY, ignore_errors=True)
print("demo cleaned")

merged model at: _demo_merged
demo cleaned


## 真实使用说明

| 组件 | mock(本 demo) | 真实 |
|------|--------------|------|
| 决策引擎 | MockDecisionEngine | `backend="laya"`(421M 本地) 或 `backend="service"`(Jeff-serve/OneJev) |
| 微调骨干 | tiny GPT2 | OneJev-0.8B / Jeff-0.8B / Laya(py3.9+) |

环境约束：laya 真实推理需 py3.9+（transformers>=4.48 支持 ModernBERT）。